# Polaris API → VictoriaLogs log-coverage

**What this answers:** for every Polaris API, *if something went wrong on this endpoint
tomorrow, would there be a record of it?*

It is a **log-generating harness**, not an API client and not a functional test suite. Its
output is a statement about observability, not about Polaris correctness — and the answer is
"no" for ten endpoints before it starts. Demonstrating that concretely is the point.

Built from `log-coverage/PLAN-log-coverage.md`, which amends
`~/hynix/local-k8s/POLARIS-API-LOG-COVERAGE-NOTEBOOK.md`. Read the plan first; this notebook
does not restate it.

## Before you run this

```bash
kubectl -n logging       port-forward svc/vlsingle-victoria-logs-single-server 9428:9428
kubectl -n datahub-hynix port-forward deploy/fb-polaris-shipper 2020:2020   # metrics only
```

`local` env only. It creates and deletes a catalog, principals, roles, namespaces, tables and
views — **the cleanup DELETEs are themselves part of the test**, so cell 9 is not optional
tidying, it is a measurement.

## Guards this run needs, and why

| guard | why |
|---|---|
| **do not trip the HPA** | Polaris scales to 3 replicas at 80% CPU, all appending to one `polaris.log` (`local-k8s` #8). The ×20 loops below are two orders of magnitude below that. Replica count is recorded before and after; a change **invalidates the run**. |
| **watch the shipper pod** | rule 6's dedup state is per-pod and in memory. A restart mid-run re-logs the first hit per table, so "exactly one" stops being an invariant. |
| **never print a token or a secret** | notebooks get committed. |
| **do not change Polaris configuration** | `local-k8s` #11/#12: it works and ships continuously, and the `quarkus.log.file.enabled=false` discrepancy is recorded as *unexplained, not pursued*. |

## The expected column is not a table anyone typed

`log_coverage.Policy` extracts `polaris_noise_filter` out of the **deployed**
`logging/fb-values.yaml` and runs it under a real Lua interpreter. If that file or a Lua
binary is missing, cell 0 aborts and says the expected column is unavailable — there is
deliberately no Python re-implementation to fall back on.

## 0 — Preflight

Aborts loudly. A coverage matrix taken from a half-working pipeline is worse than none.

In [12]:
import pathlib, shutil, subprocess, sys, time
from datetime import datetime, timezone

_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from polaris_test_utils import *          # noqa: F401,F403
import polaris_test_utils as ptu
import vlogs
import log_coverage as lc

init_env("local")
require_not_prod("the log-coverage drive creates and deletes catalogs, principals and roles")

RUN = str(int(time.time()))
NB_PRINCIPAL, NB_PROLE = f"nb_{RUN}_principal", f"nb_{RUN}_prole"
STARTED = datetime.now(timezone.utc)
V = vlogs.VLogs(ptu.VLOGS_URL)
PRE, FAIL = {}, []


def sh(*args):
    """kubectl, when it is there. Returns stdout or None -- never raises."""
    if not shutil.which(args[0]):
        return None
    try:
        r = subprocess.run(args, capture_output=True, text=True, timeout=20)
        return r.stdout.strip() if r.returncode == 0 else None
    except Exception:
        return None


try:
    ROOT_TOK = root_token()
    PRE["polaris"] = f"OK   {ptu.POLARIS_URL}"
except Exception as exc:
    ROOT_TOK = None
    FAIL.append(f"Polaris: {type(exc).__name__}: {exc}")

ok, detail = V.ping()
PRE["victorialogs"] = f"{'OK  ' if ok else 'DOWN'} {ptu.VLOGS_URL}  ({detail})"
if not ok:
    FAIL.append(
        f"VictoriaLogs unreachable at {ptu.VLOGS_URL} -- "
        "kubectl -n logging port-forward svc/vlsingle-victoria-logs-single-server 9428:9428"
    )

try:
    POLICY = lc.load_policy(ptu.FB_VALUES_PATH)
    PRE["retention policy"] = f"OK   {POLICY.source}  sha256={POLICY.sha256[:16]}"
except lc.PolicyUnavailable as exc:
    POLICY = None
    FAIL.append(f"expected column UNAVAILABLE: {exc}")

# IS THE POLICY IN THE FILE THE POLICY THAT IS RUNNING? This repo's most expensive
# failure is configuration written and never applied, and this notebook's first run
# (2026-09-04) found exactly that: fb-values.yaml gained polaris_noise_filter at
# 08:26:18Z on 2026-09-03 and the shipper pod had been up since 08:04:06Z, so the
# parser was live, the policy was not, and NOTHING was dropped. Check before driving,
# rather than inferring it from the shape of the results afterwards.
ALLOW_UNDEPLOYED_POLICY = False      # set True to drive anyway, deliberately
POLICY_LIVE, POLICY_LIVE_DETAIL = None, "policy not loaded"
if POLICY is not None:
    _cm = sh("kubectl", "-n", "datahub-hynix", "get", "cm",
             "-l", "app.kubernetes.io/instance=fb-polaris-shipper", "-o", "yaml")
    POLICY_LIVE, POLICY_LIVE_DETAIL = lc.deployed_policy_status(_cm, POLICY)
PRE["policy DEPLOYED?"] = {True: "YES ", False: "NO  ", None: "??  "}[POLICY_LIVE] + POLICY_LIVE_DETAIL.splitlines()[0]
if POLICY_LIVE is False and not ALLOW_UNDEPLOYED_POLICY:
    FAIL.append(
        "THE RETENTION POLICY IS NOT DEPLOYED. A coverage matrix taken now is a "
        "statement about a pipeline nobody is running.\n    " + POLICY_LIVE_DETAIL
    )

# Do numeric LogsQL filters actually match? `fb-values.yaml` sets `type_int_key
# http_status response_size` so they should. This cannot be answered by looking at a
# returned record's JSON type -- VictoriaLogs hands every field back as a string
# whatever it indexed -- so ask the operational question instead.
NUMERIC_STATUS, NUMERIC_DETAIL = (None, "VictoriaLogs unreachable")
if ok:
    NUMERIC_STATUS, NUMERIC_DETAIL = vlogs.numeric_status_filters_work(V)
PRE["numeric status filters"] = (
    {True: "WORK  ", False: "BROKEN", None: "?     "}[NUMERIC_STATUS] + " " + NUMERIC_DETAIL
)

FB_BEFORE, FB_ERR = vlogs.fluentbit_metrics(ptu.FB_METRICS_URL)
PRE["fluent-bit drop_records"] = FB_ERR or f"{vlogs.drop_records_total(FB_BEFORE):,.0f} (before)"

SHIPPER_BEFORE = sh(
    "kubectl", "-n", "datahub-hynix", "get", "pods",
    "-l", "app.kubernetes.io/instance=fb-polaris-shipper",
    "-o", "jsonpath={range .items[*]}{.metadata.name}@{.status.startTime}{end}",
) or "kubectl unavailable -- a shipper restart mid-run will NOT be detectable"
HPA_BEFORE = sh(
    "kubectl", "-n", "datahub-hynix", "get", "hpa",
    "-o", "jsonpath={range .items[*]}{.metadata.name}={.status.currentReplicas}{end}",
) or "kubectl unavailable -- HPA scaling will NOT be detectable"
PRE["shipper pod (before)"] = SHIPPER_BEFORE
PRE["polaris replicas (before)"] = HPA_BEFORE

for k, v in PRE.items():
    print(f"  {k:26} {v}")
if FAIL:
    raise RuntimeError(
        "PREFLIGHT FAILED:\n  - " + "\n  - ".join(FAIL)
    )
print(f"\n  run={RUN}   principal={NB_PRINCIPAL}   started={STARTED.isoformat()}")

🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
  polaris                    OK   http://192.168.139.2:8181
  victorialogs               OK   http://localhost:9428  (HTTP 200)
  retention policy           OK   /Users/kade/hynix/local-k8s/logging/fb-values.yaml  sha256=89aa2624f1f56198
  policy DEPLOYED?           YES the running ConfigMap carries this exact script
  numeric status filters     WORK   http_status:>=400 -> 164;  http_status:"404" -> 116
  fluent-bit drop_records    fluent-bit metrics unreachable at http://localhost:2020 (ConnectionError: HTTPConnectionPool(host='localhost', port=2020): Max retries exceeded with url: /api/v1/metrics (Caused by NewConnectionError("HTTPConnection(host='localhost', port=2020): Failed to establish a new connection: [Errno 61] Connection refused")))
  shipper pod (befo

## 0b — The v3 gate

Cell 0 proves the running ConfigMap matches the **file**. It cannot prove the file is policy
v3, and it says nothing about the report. Both are aborts, and neither can be inferred from the
shape of the results afterwards — which is the mistake run 1 existed to stop repeating.

`WINDOW_SECONDS` is read here off the deployed script and every wait below derives from it.
**Nothing in this notebook hardcodes 1800**: redeploy at 30 for a fast run and the assertions
stay honest.


In [ ]:
import re

GATE, GATE_FAIL = {}, []

# Comments must be stripped before grepping. The v3 script carries a comment saying the dedup
# machinery was REMOVED, and a naive search reads that as the machinery still being present --
# which it did on 2026-09-04, and cost a round trip.
_body = re.sub(r"--\[\[.*?\]\]", "", POLICY.script, flags=re.S)
_body = "\n".join(re.sub(r"--.*$", "", ln) for ln in _body.splitlines())
for marker, want in (("RESOURCE_PATTERNS", True), ("MGMT_PREFIX", True),
                     ("DEDUP_MAX_KEYS", False), ("_now_override", True)):
    here = marker in _body
    GATE[f"v3 marker {marker}"] = f"{'OK  ' if here == want else 'WRONG'} present={here} want={want}"
    if here != want:
        GATE_FAIL.append(f"{marker}: present={here}, policy v3 expects {want}")

# THE constant. Read, never assumed -- and `Policy.report_windows` refuses a caller that
# disagrees with it, because the filter indexes windows with its own value and a mismatch
# crosses no boundary, returns no report, and looks like an empty result rather than an error.
WINDOW_SECONDS = POLICY.window_seconds
TICK_SECONDS = POLICY.tick_seconds
GATE["WINDOW_SECONDS (deployed)"] = f"{WINDOW_SECONDS}s"
GATE["dummy Interval_Sec"] = f"{TICK_SECONDS}s" if TICK_SECONDS else "NOT FOUND in config.inputs"

# The tick rate is not the report period -- but it bounds two things the period cannot. It
# bounds the STARTUP BLIND SPOT (records before the first tick are counted into no window), and
# at tick >= window a late tick moves the window index by TWO: the filter reports the window it
# was holding and the one in between never existed. No single record shows that; only a gap
# between consecutive window_starts does, which is what cell 14 looks for.
if TICK_SECONDS:
    _per = WINDOW_SECONDS / TICK_SECONDS
    GATE["ticks per window"] = f"{_per:.0f}"
    if TICK_SECONDS >= WINDOW_SECONDS:
        GATE_FAIL.append(
            f"Interval_Sec {TICK_SECONDS} >= WINDOW_SECONDS {WINDOW_SECONDS}: a late tick "
            "skips a whole window, and the skipped window's records are attributed to the "
            "previous one. Lower the dummy INPUT's Interval_Sec.")
    elif _per < 4:
        GATE["ticks per window"] += "   WARN: little margin for scheduling jitter"

REPORTS_SEEN = V.count(vlogs.app_report(), start="24h")
GATE["report records (24h)"] = REPORTS_SEEN
if REPORTS_SEEN == 0:
    GATE_FAIL.append("no report records at all -- the polaris.report tick is not reaching "
                     "the filter, or the report OUTPUT is not wired to VictoriaLogs")

TICK_LEAK = V.count(vlogs.tick_leak(), start="24h")
GATE["raw tick leak (want 0)"] = TICK_LEAK
if TICK_LEAK:
    GATE_FAIL.append(f"{TICK_LEAK} raw ticks reached VictoriaLogs -- the filter should swallow "
                     "every tick inside a window; this is ~2,880 junk records a day")

# THE GATE. A window with access_seen == 0 has no resource or principal rows to split INTO, so
# it cannot answer the question -- that is not a failure, it is an unproven gate, and the run
# below produces the traffic that settles it.
BUSY = V.latest_summary(with_traffic=True, since="24h")
if BUSY is None:
    GATE["array split"] = (f"UNPROVEN -- no window with traffic yet; next boundary in "
                           f"{V.seconds_to_boundary(WINDOW_SECONDS):.0f}s. Cell 11 settles it.")
else:
    _ws = BUSY["window_start"]
    SPLIT = V.report_types(window_start=_ws, since="24h")
    GATE["array split"] = f"{SPLIT}  (window {_ws})"
    if sorted(SPLIT) != ["principal", "resource", "summary"]:
        GATE_FAIL.append(
            f"the filter's array return did not split into three report_types: {SPLIT}. "
            "One record carrying numeric-keyed fields means the schema these tests target "
            "does not exist, and the fix is in local-k8s/logging/fb-values.yaml, not here.")
    else:
        _bad = lc.check_invariants(V.report_window(_ws, since="24h"))
        GATE["stored window invariants"] = "OK" if not _bad else _bad
        if _bad:
            GATE_FAIL.append("a STORED window violates its own schema: " + "; ".join(_bad))

for k, v in GATE.items():
    print(f"  {k:28} {v}")
if GATE_FAIL:
    raise RuntimeError("v3 GATE FAILED:\n  - " + "\n  - ".join(GATE_FAIL))


## 1 — Which correlation key does this run get?

`logging.requestIdHeaderName: Polaris-Request-Id` **[from values]** says the client can supply
the request id, which would give exact per-call correlation across the access log *and* the
application log lines the request produced. Whether Polaris honours it is **[assumed]** and is
the one thing the plan could not settle from the repo.

Settle it in minute one, not hour three. If it fails, everything below still runs — the
fallback is `user_principal_name`, `%u` from the access log — but the matrix becomes per-run
rather than per-call, and the report has to say so.

In [2]:
from polaris_rest import PolarisREST

probe_rid = lc.request_id(RUN, 0, "probe")
adm_pc = PolarisREST(ptu.POLARIS_URL, ptu.REALM, token=ROOT_TOK,
                     extra_headers={"Polaris-Request-Id": probe_rid})

# GET /v1/catalogs is rule 7 ("everything else -> keep"), so it is stored on every call --
# the cheapest possible probe that is guaranteed not to be deduplicated or dropped.
r = adm_pc.list_catalogs()
echoed = r.headers.get("Polaris-Request-Id")
adm_pc.extra_headers = {}

poll = V.poll_until(
    vlogs.and_(vlogs.app_polaris(), vlogs.field_eq("mdc.requestId", probe_rid)),
    expect_at_least=1, timeout=45, interval=1, start="15m",
)
CORRELATION = "exact" if poll.satisfied else "fallback"

print(f"  sent     Polaris-Request-Id: {probe_rid}")
print(f"  echoed   Polaris-Request-Id: {echoed}   ({'honoured' if echoed == probe_rid else 'REWRITTEN by the server'})")
print(f"  found in VictoriaLogs: {len(poll)} record(s) after {poll.waited_s:.1f}s")
print(f"\n  CORRELATION MODE = {CORRELATION.upper()}")
if CORRELATION == "fallback":
    print("  -> mdc.requestId does not round-trip. Per-call rows become per-endpoint rows;")
    print("     the run is selected by user_principal_name and the time window instead.")
    print("     This is a finding in its own right -- record it in the report.")
INGEST_LAG_S = poll.waited_s

  sent     Polaris-Request-Id: nb-1788498536-000-probe
  echoed   Polaris-Request-Id: nb-1788498536-000-probe   (honoured)
  found in VictoriaLogs: 8 record(s) after 2.0s

  CORRELATION MODE = EXACT


## 2 — The denominator: three sources, kept apart

`PLAN` §2.2. Polaris 8181/8182 serves **no** OpenAPI document (measured 2026-08-31), so the
spec has to be vendored — run `./fetch_specs.sh` once. The three sources are unequal and are
never collapsed into a percentage:

| source | what it proves | what it cannot |
|---|---|---|
| **spec** | this VERSION names the endpoint | that this BUILD serves it — generic tables and policies are feature-flagged |
| **captured** | this DEPLOYMENT served it | anything about what was never called: a call log is a lower bound |
| **driven** | this notebook calls it | — |

In [3]:
import api_surface as surf
from iceberg_rest import build_create_table_payload, build_scan_report, build_schema

HERE = pathlib.Path.cwd()
SPEC, SPEC_FILES = lc.spec_inventory(HERE / "spec")
CAPTURED, CAPTURED_FROM = lc.captured_inventory(
    HERE.parent / "diagnostics" / "api-sql-profile" / "reports"
)

if not SPEC:
    print("  spec      NOT VENDORED -- run ./fetch_specs.sh. The 'candidate' column will be")
    print("            absent, and an endpoint this build does not serve cannot be told")
    print("            apart from one nobody has called.")
else:
    for f in SPEC_FILES:
        print(f"  spec      {f['file']:38} {f['paths']:3} paths  sha256={f['sha256'][:12]}")
    print(f"  spec      {len(SPEC)} operations")
print(f"  captured  {len(CAPTURED)} operations from {CAPTURED_FROM.name if CAPTURED_FROM else 'NOTHING -- no parsable matrix report'}")
print(f"  driven    43 operations (api_surface.operations)")

# The verdict is printed here, before anything is driven, because "confirmed gap" is a
# fact about the harness and does not need this run to establish it.
import collections as _c

DENOM = lc.coverage_rows(SPEC, CAPTURED, lc.driven_inventory(surf.operations(
    surf.SurfaceContext(), build_create_table_payload, build_scan_report)))
print("\n  " + "  ".join(f"{k}={v}" for k, v in
                          _c.Counter(r["verdict"] for r in DENOM).most_common()))
_gaps = [r for r in DENOM if r["verdict"] == "confirmed gap"]
if _gaps:
    print("  CONFIRMED GAPS -- this cluster served these and the notebook does not call them:")
    for r in _gaps[:20]:
        print(f"    {r['method']:6} {r['path']}")

  spec      polaris-management-service.yml          17 paths  sha256=f5d18f7310a2
  spec      rest-catalog-open-api.yaml              18 paths  sha256=91786328d812
  spec      63 operations
  captured  40 operations from doc-api-sql-matrix-admin-explained-20260903-142636.md
  driven    43 operations (api_surface.operations)

  driven=40  candidate=24


## 3 — Happy path, in dependency order

`api_surface.operations()` is the 43-operation surface, in issue order, and the order is
load-bearing: three chains depend on it (namespace create→drop, table create→rename→drop,
view create→rename→drop) and the management block creates a principal, a principal-role and a
catalog-role before granting on them and deleting them.

Driven with `log_coverage.drive_tagged`, **not** `api_surface.drive`: that one is built around
a SQL `Tracer` and aborts when the capture reads nothing, which is correct for the API→SQL
audit and meaningless here.

`elapsed_ms` is measured client-side because there is nowhere else to get it — the access-log
pattern has no `%D`, so **nothing in VictoriaLogs can say how long any request took**. These
numbers are the evidence for adding it.

In [4]:
from iceberg_rest import IcebergREST

adm_ic = IcebergREST(ptu.POLARIS_URL, ptu.REALM, token=ROOT_TOK)
SCHEMA = build_schema([(1, "id", "long", True), (2, "val", "string", False)])

# `apiprofile*` is the stem teardown_fixture sweeps, so keep it -- and build the
# fixture directly rather than via .stamped(), whose timestamp would not be this
# run's and would make a stranded catalog un-attributable.
FX = surf.ProbeFixture(prefix=f"apiprofile{RUN}")
surf.setup_fixture(FX, adm_pc, adm_ic, ptu.BUCKET, ptu.MINIO_ENDPOINT, SCHEMA,
                   build_create_table_payload)
print(f"  fixture ready: {FX.cat}")

IDENTITY, pc, ic, DRIVE_PRINCIPAL, IDENTITY_NOTES = lc.elect_drive_identity(
    adm_pc, adm_ic, ptu.POLARIS_URL, ptu.REALM, NB_PRINCIPAL, NB_PROLE, fixture=FX
)
for n in IDENTITY_NOTES:
    print(f"  identity  {n}")

CTX = FX.context(ic, pc, adm_ic, SCHEMA)
CTX.adm = adm_pc          # so the fixture's own calls are tagged too
OPS = surf.operations(CTX, build_create_table_payload, build_scan_report)

CALLS = lc.drive_tagged(OPS, CTX, RUN)
print(f"\n  drove {len(CALLS)} operations as {DRIVE_PRINCIPAL}")

import collections
print("  status distribution:", dict(collections.Counter(c["status"] for c in CALLS)))
_err = [c for c in CALLS if c["error"]]
if _err:
    print(f"  {len(_err)} HARNESS ERRORS (not refusals -- these are the harness failing):")
    for c in _err[:10]:
        print(f"    {c['label']}: {c['error']}")

  fixture ready: apiprofile1788498536_cat
  identity  driving as nb_1788498536_principal (scope=PRINCIPAL_ROLE:ALL)

  drove 43 operations as nb_1788498536_principal
  status distribution: {200: 22, 204: 11, 500: 2, 404: 1, 201: 6, 403: 1}


## 4 — Negative cases

Rule 3 (`http_status >= 400` → keep) is the pipeline's real safety net — **not** rule 1, since
a failed request is usually not an ERROR log line at all, just an access-log line with a 4xx
and nothing else. These drive it.

The 500 is `create_catalog_no_endpoint` (`error-cases/09_500_null_pointer`): a catalog missing
`endpointInternal` throws a NullPointerException on table create. Cheapest reliable 500 in
this repo, and reversible — the catalog is deleted immediately after. It is here to answer the
plan's highest-value unknown: **do exception stack traces survive the pipeline?**

The timeout case is honest about its own limit. A client-side timeout does not stop Quarkus
completing the response and writing the access-log line, so it cannot produce the blind spot
being described. The real blind spot is a **server** hang, which cannot be provoked from here.

In [5]:
import requests

NEG = []
seq = 900


def neg(label, fn, method="", path=""):
    global seq
    seq += 1
    row = lc.call_once([pc, ic, adm_pc, adm_ic], RUN, seq, label, fn, method, path)
    NEG.append(row)
    print(f"  {row['status'] if row['status'] is not None else row['error'][:40]:>6}  {label}")
    return row


# 404 -- a table that is not there. Rule 3 keeps EVERY one of these; rule 6 never sees it.
neg("neg.404_missing_table", lambda: ic.load_table(FX.cat, FX.ns, "definitely_not_here"),
    "GET", "")

# 401 -- a real client id with the wrong secret. The token endpoint's SUCCESS is dropped by
# rule 5; its FAILURE is kept by rule 3. So "who failed to authenticate" is answerable and
# "who authenticated" is not.
neg("neg.401_bad_secret", lambda: adm_pc.get_token(ptu.ROOT_CLIENT, "not-the-secret"),
    "POST", "")

# 403 -- a principal with a role and no grants at all.
DENIED = f"nb_{RUN}_denied"
DENIED_ROLE = f"nb_{RUN}_denied_role"
denied_ic = None
try:
    _cid, _sec = lc.provision_run_principal(adm_pc, DENIED, DENIED_ROLE,
                                            fixture=None, service_admin=False)
    _r = adm_pc.get_token(_cid, _sec, scope=f"PRINCIPAL_ROLE:{DENIED_ROLE}")
    if _r.status_code == 200:
        denied_ic = IcebergREST(ptu.POLARIS_URL, ptu.REALM, token=_r.json()["access_token"])
        # denied_ic must be tagged too. On 2026-09-04 it was not in the client list, so
        # the 403 went out with no Polaris-Request-Id, could not be found afterwards,
        # and the matrix reported "EXPECTED STORED, ABSENT" for a record that was
        # almost certainly there -- a harness gap wearing a finding's clothes.
        seq += 1
        row = lc.call_once([denied_ic], RUN, seq, "neg.403_no_privilege",
                           lambda: denied_ic.load_table(FX.cat, FX.ns, FX.tbl), "GET", "")
        NEG.append(row)
        print(f"  {row['status']:>6}  neg.403_no_privilege")
    else:
        print(f"  ---     403 case skipped: token for {DENIED} -> {_r.status_code}")
except Exception as exc:
    print(f"  ---     403 case skipped: {type(exc).__name__}: {exc}")

# 409 -- create a namespace that already exists.
neg("neg.409_already_exists", lambda: ic.create_namespace(FX.cat, FX.ns), "POST", "")

# 400 -- a create-table body with no schema.
neg("neg.400_malformed_body", lambda: ic.create_table(FX.cat, FX.ns, {"name": "no_schema"}),
    "POST", "")

# 405 -- a method the endpoint does not implement.
neg("neg.405_method_not_allowed",
    lambda: requests.request("DELETE", f"{ptu.POLARIS_URL}/api/catalog/v1/config",
                             headers=ic._h(), timeout=15),
    "DELETE", "")

# 500 -- NullPointerException on table create against a catalog with no endpointInternal.
NPE_CAT = f"nb{RUN}npe"
try:
    _r = create_catalog_no_endpoint(name=NPE_CAT, token=ROOT_TOK)
    print(f"  {_r.status_code:>6}  neg.500_setup_catalog_without_endpointInternal")
    ic.create_namespace(NPE_CAT, "npe_ns")
    neg("neg.500_null_pointer",
        lambda: ic.create_table(NPE_CAT, "npe_ns",
                                build_create_table_payload("npe_tbl", SCHEMA)),
        "POST", "")
finally:
    try:
        adm_pc.delete_catalog(NPE_CAT, purge=True)
    except Exception as exc:
        print(f"  ---     could not remove {NPE_CAT}: {exc}")

# timeout -- attempted, and expected NOT to reproduce the blind spot. Recorded either way.
neg("neg.client_timeout",
    lambda: requests.get(f"{ptu.POLARIS_URL}/api/management/v1/catalogs",
                         headers=adm_pc._h(), timeout=0.001),
    "GET", "/api/management/v1/catalogs")

# 429 is UNREACHABLE: rateLimiter.type is no-op [from values]. Recorded as not callable,
# with the reason. An honest gap beats a fabricated pass.
NOT_CALLABLE = {
    "429 rate limited": "rateLimiter.type: no-op -- the limiter cannot produce a 429",
    "OPA authorization paths": "opa.enabled: false -- those code paths are dead here",
    "federated catalog endpoints": "no federated catalog is configured on this cluster",
    "a server-side hang": "cannot be provoked from a client; the access log is written when "
                          "the response is, so a hang leaves NO line at all and this pipeline "
                          "is blind to it",
}

     404  neg.404_missing_table
     401  neg.401_bad_secret
     403  neg.403_no_privilege
     409  neg.409_already_exists
     400  neg.400_malformed_body
     405  neg.405_method_not_allowed
     201  neg.500_setup_catalog_without_endpointInternal
     200  neg.500_null_pointer
  ReadTimeout: HTTPConnectionPool(host='19  neg.client_timeout


## 5 — Policy probes

Deliberate tests of the **v3** branches. Every expected count comes from `Policy.predict()` —
the deployed Lua, run in issue order — so the expected column cannot drift from what ships.

v2's probes asked how dedup behaved. **There is no dedup in v3**, so those questions have no
answer and have been replaced rather than left to pass vacuously. The interesting question is
now *counted or kept*, and the two halves of the v3 trade are probed explicitly: every
management mutation is **stored** (the audit hole v2 left open, closed), and every successful
read is **counted only** (what it cost).

The ordering in the first block is load-bearing: read the table successfully, **then** write
it, **then** provoke a 403 on it, so one resource row carries `requests`, `reads`, `writes` and
`errors` all non-zero. An error on a resource never read successfully lands in `__other__`
instead of creating a key.


In [ ]:
PROBE_CAT, PROBE_NS, PROBE_TBL, PROBE_VIEW = FX.cat, FX.ns, FX.tbl, FX.view
PROBES, seq = [], 1000


def probe(name, n, fn, method, path_for_prediction):
    """Run `fn` n times, tagged. The expectation is computed in cell 8, not here.

    Predicted across the WHOLE run in issue order rather than per probe. Under
    v2 that mattered because rule 6 carried per-day state; under v3 the filter
    keeps no keep/drop state at all, but the REPORT counters do accumulate
    across the run, so the scope still has to be the run.
    """
    global seq
    rows = []
    for i in range(n):
        seq += 1
        rows.append(lc.call_once([pc, ic, adm_pc, adm_ic], RUN, seq, f"probe.{name}",
                                 fn, method, path_for_prediction))
    PROBES.append({"probe": name, "calls": n, "rows": rows})
    codes = sorted({r["status"] for r in rows})
    print(f"  {name:46} calls={n:<3} status={codes}")
    return rows


CATP = f"/api/catalog/v1/{PROBE_CAT}"
MGMTP = "/api/management/v1"
TBL_PATH = f"{CATP}/namespaces/{PROBE_NS}/tables/{PROBE_TBL}"

# --- one resource row with requests, reads, writes AND errors all non-zero ---
# ORDER IS LOAD-BEARING. Read it successfully FIRST: `touch_resource(key, kind, create)` is
# called with create=false for an error, so a 403 on a table nobody has read yet lands in
# __other__ and never creates the key. Read, then write, then fail.
probe("read the table (rule 6, counted)", 3,
      lambda: ic.load_table(PROBE_CAT, PROBE_NS, PROBE_TBL), "GET", TBL_PATH)
probe("write the table (rule 5b, counted; writes+1)", 1,
      lambda: ic.commit_table(PROBE_CAT, PROBE_NS, PROBE_TBL,
                              [{"action": "set-properties", "updates": {"nb.run": RUN}}]),
      "POST", TBL_PATH)
if denied_ic is not None:
    probe("403 on the table (rule 3, KEPT; errors+1)", 1,
          lambda: denied_ic.load_table(PROBE_CAT, PROBE_NS, PROBE_TBL), "GET", TBL_PATH)
else:
    print("  !! no unauthorized client -- the table row will carry errors=0 and cell 12's "
          "four-non-zero-fields assertion is VOID, not failed")
# `/metrics` is driven by the happy path (api_surface's iceberg.report_metrics builds a real
# ScanReport). Cell 12 asserts what matters about it: its requests land on the TABLE's row and
# NO row exists for the /metrics path. v2 emitted two rows for one table.

# --- rule 6: successful reads are counted only. All of these were STORED under v2. ---
probe("same table GET x20 (counted)", 20,
      lambda: ic.load_table(PROBE_CAT, PROBE_NS, PROBE_TBL), "GET", TBL_PATH)
probe("HEAD the same table (counted)", 1,
      lambda: ic.head_table(PROBE_CAT, PROBE_NS, PROBE_TBL), "HEAD", TBL_PATH)
probe("table LIST x20 (counted; v2 stored 20)", 20,
      lambda: ic.list_tables(PROBE_CAT, PROBE_NS),
      "GET", f"{CATP}/namespaces/{PROBE_NS}/tables")
probe("GET /config (counted; v2 stored every one)", 3,
      lambda: ic.get_config(), "GET", "/api/catalog/v1/config")
probe("management read (counted)", 1,
      lambda: adm_pc.list_principals(), "GET", f"{MGMTP}/principals")
probe("view read (resource_kind=view)", 1,
      lambda: ic.load_view(PROBE_CAT, PROBE_NS, PROBE_VIEW),
      "GET", f"{CATP}/namespaces/{PROBE_NS}/views/{PROBE_VIEW}")
probe("namespace read (resource_kind=namespace)", 1,
      lambda: ic.load_namespace(PROBE_CAT, PROBE_NS),
      "GET", f"{CATP}/namespaces/{PROBE_NS}")

# --- rule 3: errors are kept, ALL of them, no cap -- and they do not create a key ---
probe("missing table GET x20 (all KEPT, -> __other__)", 20,
      lambda: ic.load_table(PROBE_CAT, PROBE_NS, "nope"),
      "GET", f"{CATP}/namespaces/{PROBE_NS}/tables/nope")

# --- rule 5a: THE V2 AUDIT HOLE, now closed. Every one of these must be STORED. ---
ASYM = f"nb_{RUN}_asym"
probe("create a principal (rule 5a, now KEPT)", 1,
      lambda: adm_pc.create_principal(ASYM), "POST", f"{MGMTP}/principals")
probe("reset its credentials (rule 5a, now KEPT)", 1,
      lambda: adm_pc.reset_principal_credentials(ASYM),
      "POST", f"{MGMTP}/principals/{ASYM}/reset")
probe("delete that principal (rule 4)", 1,
      lambda: adm_pc.delete_principal(ASYM), "DELETE", f"{MGMTP}/principals/{ASYM}")

# --- rule 5b: every other successful POST is counted only ---
probe("successful token x3 (counted)", 3,
      lambda: adm_pc.get_token(ptu.ROOT_CLIENT, ptu.ROOT_SECRET),
      "POST", "/api/catalog/v1/oauth/tokens")
RENAMED = f"{PROBE_TBL}_renamed"
probe("rename a table (counted)", 1,
      lambda: ic.rename_table(PROBE_CAT, PROBE_NS, PROBE_TBL, PROBE_NS, RENAMED),
      "POST", f"{CATP}/tables/rename")
probe("rename it back (counted)", 1,
      lambda: ic.rename_table(PROBE_CAT, PROBE_NS, RENAMED, PROBE_NS, PROBE_TBL),
      "POST", f"{CATP}/tables/rename")


## 6 — Where is a PUT's request body logged?

Rule 4 keeps every PUT/DELETE/PATCH, but **the request body is not in the access log**. A PUT
that changes a role's privileges is recorded as *"a PUT happened to this path, and it returned
200"*. For an audit trail of grants that is the whole point, and it is missing.

So: grant a privilege, then search VictoriaLogs for *any* record carrying the privilege name.
Then look at the other channel. The `eventListener` writes audit events to a PostgreSQL table
(`type: persistence-in-memory-buffer`, 5s flush, 1000-event buffer **[from values]**) — a
second audit channel that does not go through VictoriaLogs at all. If that is where "what
changed" lives, **it changes where you go during an incident**, and the report has to say so.

In [7]:
GRANTED_PRIV = "TABLE_READ_DATA"
seq += 1
grant_row = lc.call_once([pc, ic, adm_pc, adm_ic], RUN, seq, "probe.grant_privilege",
                         lambda: adm_pc.grant_privilege(FX.cat, "catalog_admin", GRANTED_PRIV),
                         "PUT", f"/api/management/v1/catalogs/{FX.cat}/catalog-roles/catalog_admin/grants")
print(f"  grant {GRANTED_PRIV} -> {grant_row['status']}")

time.sleep(max(2.0, INGEST_LAG_S))
priv_hits = V.query(vlogs.and_(vlogs.app_polaris(), f"_msg:{vlogs.quote(GRANTED_PRIV)}"),
                    start="10m", limit=50)
print(f"  records in VictoriaLogs carrying the privilege NAME: {len(priv_hits)}")
if not priv_hits:
    print("  -> the pipeline cannot answer 'who granted what'. The PUT is stored; the body is not.")

# Polaris creates its objects in POLARIS_SCHEMA, not `public` -- looking there reported
# "no events table in this metastore" on 2026-09-04 for a table that may well exist.
EVENTS = {"reachable": False, "rows": None, "sample": None, "note": None, "schema": None}
try:
    import psycopg2
    with psycopg2.connect(**ptu.PG_CONFIG) as conn:
        EVENTS.update(lc.events_table(conn))
        EVENTS["reachable"] = EVENTS.get("rows") is not None
except Exception as exc:
    EVENTS["note"] = f"{type(exc).__name__}: {exc}"

if EVENTS["reachable"]:
    print(f"  found `events` in schema {EVENTS['schema']}")
    print(f"  eventListener `events` table: {EVENTS['rows']} rows -- a SECOND audit channel,")
    print("  and it does not pass through VictoriaLogs at all.")
    for row in EVENTS["sample"] or []:
        print("   ", {k: str(v)[:60] for k, v in row.items()})
else:
    print(f"  events table not read: {EVENTS['note']}")
    print("  (needs a port-forward: kubectl -n datahub-hynix port-forward "
          "pod/benchmarks-postgresql-postgresql-ha-postgresql-0 5433:5432)")

  grant TABLE_READ_DATA -> 201
  records in VictoriaLogs carrying the privilege NAME: 3
  found `events` in schema polaris_schema
  eventListener `events` table: 2169 rows -- a SECOND audit channel,
  and it does not pass through VictoriaLogs at all.
    {'realm_id': 'POLARIS', 'catalog_id': 'user51_catalog', 'event_id': 'cb94b83a-c67c-4f4a-a62a-8254892dc2d7', 'request_id': '3923fb9f-f6ae-4f2e-8c15-15420a28bf75_0000000000000000960', 'event_type': 'AfterCreateCatalogEvent', 'timestamp_ms': '1787533896559', 'principal_name': 'root', 'resource_type': 'CATALOG', 'resource_identifier': 'user51_catalog', 'additional_properties': '{}'}
    {'realm_id': 'POLARIS', 'catalog_id': 'user1_catalog', 'event_id': '3a6cd508-12f7-4622-b3f2-c13586cbe80c', 'request_id': '3923fb9f-f6ae-4f2e-8c15-15420a28bf75_0000000000000000968', 'event_type': 'AfterCreateCatalogEvent', 'timestamp_ms': '1787533961568', 'principal_name': 'root', 'resource_type': 'CATALOG', 'resource_identifier': 'user1_catalog', 'additiona

## 7 — Collection

Ingest is asynchronous, so this polls rather than sleeping and hoping — and it deduplicates by
`mdc.requestId` before counting, because the shipper's tail offset DB lives on an `emptyDir`:
a `helm upgrade` replaces the pod, `Read_from_Head true` re-reads the whole file, and
VictoriaLogs does not deduplicate on ingest. **Records can legitimately appear twice.**

In [8]:
ALL_CALLS = CALLS + NEG + [r for p in PROBES for r in p["rows"]] + [grant_row]
WINDOW_START = STARTED
WINDOW_END = datetime.now(timezone.utc)

if CORRELATION == "exact":
    run_filter = vlogs.and_(vlogs.app_polaris(), f'"mdc.requestId":"nb-{RUN}-"*')
else:
    run_filter = vlogs.and_(vlogs.app_polaris(),
                            vlogs.field_eq("user_principal_name", DRIVE_PRINCIPAL))

pull = V.settle(run_filter, quiet_for=4, timeout=90, interval=2,
                start=WINDOW_START, end=WINDOW_END, limit=10000)
# Deduplicate REPLAYED RECORDS, not request ids: one call produces many records that
# share a request id (its access-log line plus every application line it emitted), and
# keying on the request id collapses them into one. See vlogs.dedup_replayed.
STORED = vlogs.dedup_replayed(pull.records)
DUPES = len(pull.records) - len(STORED)

BY_RID = {}
for rec in STORED:
    BY_RID.setdefault(rec.get("mdc.requestId"), []).append(rec)

print(f"  calls issued          {len(ALL_CALLS)}")
print(f"  records returned      {len(pull.records)}  (settled after {pull.waited_s:.0f}s)")
print(f"  after replay dedup    {len(STORED)}   [{DUPES} duplicate(s) -- a shipper replay if > 0]")
print(f"  distinct request ids  {len([k for k in BY_RID if k])}")
print(f"  records per call      {len(STORED) / max(1, len(ALL_CALLS)):.1f}")

FB_AFTER, FB_AFTER_ERR = vlogs.fluentbit_metrics(ptu.FB_METRICS_URL)
DROP_DELTA = (None if (FB_ERR or FB_AFTER_ERR)
              else vlogs.drop_records_total(FB_AFTER) - vlogs.drop_records_total(FB_BEFORE))
print(f"  fluent-bit drop delta {DROP_DELTA if DROP_DELTA is not None else FB_AFTER_ERR}")

SHIPPER_AFTER = sh("kubectl", "-n", "datahub-hynix", "get", "pods",
                   "-l", "app.kubernetes.io/instance=fb-polaris-shipper",
                   "-o", "jsonpath={range .items[*]}{.metadata.name}@{.status.startTime}{end}")
HPA_AFTER = sh("kubectl", "-n", "datahub-hynix", "get", "hpa",
               "-o", "jsonpath={range .items[*]}{.metadata.name}={.status.currentReplicas}{end}")
SHIPPER_MOVED = bool(SHIPPER_AFTER) and SHIPPER_AFTER != SHIPPER_BEFORE
HPA_MOVED = bool(HPA_AFTER) and HPA_AFTER != HPA_BEFORE
if SHIPPER_MOVED:
    print("  !! THE SHIPPER RESTARTED MID-RUN. Dedup state is per-pod and in memory, so the")
    print("     'exactly one per day' expectations below are VOID for this run, not failed.")
if HPA_MOVED:
    print("  !! POLARIS SCALED MID-RUN. Multiple replicas append to one log file "
          "(local-k8s #8) -- this run is INVALID.")

  calls issued          122
  records returned      2026  (settled after 4s)
  after replay dedup    2026   [0 duplicate(s) -- a shipper replay if > 0]
  distinct request ids  123
  records per call      16.6
  fluent-bit drop delta fluent-bit metrics unreachable at http://localhost:2020 (ConnectionError: HTTPConnectionPool(host='localhost', port=2020): Max retries exceeded with url: /api/v1/metrics (Caused by NewConnectionError("HTTPConnection(host='localhost', port=2020): Failed to establish a new connection: [Errno 61] Connection refused")))


## 8 — The coverage matrix

One row per endpoint. `expected` comes from the deployed Lua; `stored` from VictoriaLogs; the
`discrepancy` column is the only one worth reading twice.

A row where expected is *drop* and stored is 0 is **not a pass** in the ordinary sense — it is
the pipeline working exactly as configured, and the configuration being wrong. Those rows are
the finding.

In [9]:
import pandas as pd

EXPECTED = lc.expected_for(ALL_CALLS, POLICY, user=DRIVE_PRINCIPAL) if POLICY else [None] * len(ALL_CALLS)

rows = []
for call, exp in zip(ALL_CALLS, EXPECTED):
    found = BY_RID.get(call["request_id"], []) if CORRELATION == "exact" else None
    access = [r for r in (found or []) if vlogs.is_access_log(r)]
    rows.append({
        "label": call["label"],
        "method": call["method"],
        "path": (call["actual_path"] or call["path"] or "")[-58:],
        "status": call["status"],
        "ms": call["elapsed_ms"],
        "expected": (exp or {}).get("verdict") if exp else None,
        # `disposition` is the column v3 needs and v2 did not. `stored` is 0 for most calls
        # BY DESIGN now, so an expected/stored matrix mostly reads "correctly dropped" and
        # hides the interesting half: which aggregate row should have moved instead.
        "disposition": lc.disposition((exp or {}).get("verdict")) if exp else None,
        "stored": len(access) if found is not None else None,
        # app_lines is derived from THIS pull, not from a second source: `found` is every
        # record VictoriaLogs returned for the call's request id, `access` the access-log
        # subset, and the difference is the application lines the request emitted. That
        # settles the open question the previous run left ("where does app_lines come
        # from?") -- VictoriaLogs, via mdc.requestId, and nowhere else.
        "app_lines": len(found) - len(access) if found is not None else None,
    })
M = pd.DataFrame(rows)

# Which resource row and which principal row SHOULD each call have incremented? The keys come
# from the deployed classify() -- driven one path per window so the answer is the filter's own,
# never a Python re-implementation of RESOURCE_PATTERNS.
_paths = sorted({(c.get("actual_path") or lc.full_path(c["api"], c["path"])).split("?")[0]
                 for c in ALL_CALLS if c.get("status")})
KEYS = POLICY.classify_paths(_paths) if POLICY else {}
M["resource_row"] = [
    (KEYS.get((c.get("actual_path") or lc.full_path(c["api"], c["path"])).split("?")[0])
     or (None, None))[0] for c in ALL_CALLS
]
M["kind"] = [
    (KEYS.get((c.get("actual_path") or lc.full_path(c["api"], c["path"])).split("?")[0])
     or (None, None))[1] for c in ALL_CALLS
]
M["principal_row"] = DRIVE_PRINCIPAL
M["discrepancy"] = [
    "" if r.expected is None or r.stored is None
    else "" if (r.expected == "keep") == (r.stored > 0)
    else ("EXPECTED STORED, ABSENT" if r.expected == "keep" else "EXPECTED DROPPED, PRESENT")
    for r in M.itertuples()
]
pd.set_option("display.width", 200, "display.max_rows", 300, "display.max_colwidth", 60)
print(M.to_string(index=False))

print("\n--- calls that leave NO individual record: COUNTED, not lost ---")
_drop = M[M.expected == "drop"]
print(f"  {len(_drop)} calls; {int((_drop.stored.fillna(0) > 0).sum())} of them are stored anyway")
print(_drop[["label", "method", "status", "stored", "resource_row", "kind"]].to_string(index=False))

print("\n--- the v3 trade, both halves ---")
_mgmt = M[M.path.str.contains("/api/management", na=False) & M.method.isin(["POST"])]
print(f"  management POSTs driven: {len(_mgmt)};  kept: {int((_mgmt.expected == 'keep').sum())}"
      "   (v2 kept NONE of these -- a principal was created invisibly and deleted visibly,")
print("    and a credential reset left no trace at all. That is the hole v3 closes.)")
_reads = M[(M.method.isin(["GET", "HEAD"])) & (M.status.fillna(0) < 400)]
print(f"  successful reads driven: {len(_reads)};  counted only: "
      f"{int((_reads.expected == 'drop').sum())}   (what it cost: no per-call timestamp,")
print("    no per-table access frequency -- only the window aggregate.)")

print("\n--- resource_kind coverage, from the deployed classify() ---")
_kinds = sorted({k for k in M["kind"] if k})
print(f"  seen: {_kinds}")
print(f"  missing: {sorted(set(lc.RESOURCE_KINDS) - set(_kinds)) or 'none -- all six driven'}")

print("\n--- discrepancies (expected vs actual) ---")
_d = M[M.discrepancy != ""]
print("none" if _d.empty else _d.to_string(index=False))

print("\n--- probe expectations (predicted across the WHOLE run, in issue order) ---")
EXP_BY_RID = {c["request_id"]: (e or {}).get("verdict") for c, e in zip(ALL_CALLS, EXPECTED)}
for p in PROBES:
    want = sum(1 for row in p["rows"] if EXP_BY_RID.get(row["request_id"]) == "keep")
    got = (sum(len([r for r in BY_RID.get(row["request_id"], []) if vlogs.is_access_log(r)])
               for row in p["rows"]) if CORRELATION == "exact" else None)
    verdict = "" if got is None or got == want else "   <-- MISMATCH"
    print(f"  {p['probe']:44} calls {p['calls']:<3} expected {want:<3} got {got}{verdict}")

print("\n--- WARN / ERROR messages seen (source plan section 8.1) ---")
WARNS = V.query(vlogs.and_(vlogs.app_polaris(), 'level:"WARN" OR level:"ERROR"'),
                start=WINDOW_START, end=WINDOW_END, limit=1000)
DISTINCT_WARNS = {}
for w in WARNS:
    key = (w.get("level"), (w.get("_msg") or "")[:120])
    DISTINCT_WARNS[key] = DISTINCT_WARNS.get(key, 0) + 1
for (lvl, msg), n in sorted(DISTINCT_WARNS.items(), key=lambda kv: -kv[1]):
    print(f"  {lvl:5} x{n:<4} {msg}")
if not DISTINCT_WARNS:
    print("  none -- so the 'Deprecated Config' exclusion still has no message to match on.")

print("\n--- did any record carry an exception / stack trace? ---")
EXC = [r for r in WARNS if "exception" in r or "stackTrace" in str(r)]
print(f"  {len(EXC)} of {len(WARNS)} WARN/ERROR records carry an exception object")
if EXC:
    print("  ->", str(EXC[0].get("exception") or EXC[0].get("_msg"))[:300])
else:
    print("  -> NOT DEMONSTRATED. Either the 500 produced no throwable in the log, or the")
    print("     pipeline drops them. Check the raw file before concluding:")
    print("     kubectl -n datahub-hynix exec deploy/benchmarks-polaris -- "
          "grep -c stackTrace /deployments/logs/polaris.log")

                                   label method                                                       path  status    ms expected  stored  app_lines               discrepancy
                      iceberg.get_config    GET  /api/catalog/v1/config?warehouse=apiprofile1788498536_cat   200.0   8.8     keep       1         10                          
                 iceberg.list_namespaces    GET        /api/catalog/v1/apiprofile1788498536_cat/namespaces   200.0   8.3     keep       1         10                          
                  iceberg.load_namespace    GET pi/catalog/v1/apiprofile1788498536_cat/namespaces/probe_ns   200.0   8.9     keep       1          9                          
                  iceberg.head_namespace   HEAD pi/catalog/v1/apiprofile1788498536_cat/namespaces/probe_ns   204.0   9.5     keep       1          9                          
     iceberg.update_namespace_properties   POST v1/apiprofile1788498536_cat/namespaces/probe_ns/properties   200.0  12.9     

## 9 — Cleanup

The cleanup DELETEs are part of the test — they are driven inside the tagged window, not after it. A failed run must still leave the cluster usable.

In [10]:
CLEANUP = {}
CLEANUP["fixture"] = surf.teardown_fixture(FX, adm_pc, adm_ic, sweep_stale=False)
CLEANUP["run_principal"] = lc.deprovision_run_principal(adm_pc, NB_PRINCIPAL, NB_PROLE)
CLEANUP["denied_principal"] = lc.deprovision_run_principal(adm_pc, DENIED, DENIED_ROLE)
try:
    adm_pc.delete_catalog(f"nb{RUN}npe", purge=True)
except Exception:
    pass

for k, v in CLEANUP.items():
    print(f"  {k:20} {v}")

try:
    del pull
except NameError:
    pass
import gc
gc.collect()
print("\n  cleaned. Anything left behind is listed above.")

  fixture              {'fixture': {'tables': 1, 'views': 0, 'namespaces': 2, 'catalog_roles': 1, 'catalog': 204, 'failed': {}, 'builtin_role_skipped': 'catalog_admin'}, 'swept': {}}
  run_principal        []
  denied_principal     []

  cleaned. Anything left behind is listed above.


## 11 — The scheduled report: the window that saw this run

The flush report is a **scheduled summariser**, and this is where it stops being a schema and
starts being a thing that has to fire on time. A `dummy` INPUT tagged `polaris.report` ticks
every 30 s into the same Lua filter; on each window boundary the filter replaces the tick with
an *array* of records and resets its counters.

This cell waits for the boundary that closes the window holding the run, pulls its rows, and
runs `log_coverage.check_invariants` over them — the same function the oracle output is checked
with, which is what makes a difference between the two sides a pipeline finding rather than a
test artefact.

**The wait is real wall-clock**, derived from the deployed `WINDOW_SECONDS`. At 1800 it is up
to half an hour; redeploy at 30 (and drop the dummy INPUT's `Interval_Sec` with it) for a fast
run. Nothing here changes either way.

In [ ]:
REPORT = {}
_to_boundary = V.seconds_to_boundary(WINDOW_SECONDS, lag=0)
print(f"  window_seconds={WINDOW_SECONDS}  next boundary in {_to_boundary:.0f}s"
      f"  (+{INGEST_LAG_S:.0f}s measured ingest lag)")

RUN_WINDOW = V.wait_for_boundary(WINDOW_SECONDS, lag=max(8.0, INGEST_LAG_S * 3))
print(f"  window closed: {RUN_WINDOW}")

W1 = V.report_window(RUN_WINDOW, since="6h")
SPLIT = {}
for r in W1:
    SPLIT[r.get("report_type", "(none)")] = SPLIT.get(r.get("report_type", "(none)"), 0) + 1
print(f"  rows: {len(W1)}   split: {SPLIT}")

if sorted(SPLIT) != ["principal", "resource", "summary"]:
    raise RuntimeError(
        f"the array return did not split into three report_types: {SPLIT}. One record "
        "carrying numeric-keyed fields means this schema does not exist; the fix is in "
        "local-k8s/logging/fb-values.yaml, not here.")

BAD = lc.check_invariants(W1)
print("  invariants:", "OK" if not BAD else "")
for b in BAD:
    print("   !!", b)

S1 = [r for r in W1 if r["report_type"] == "summary"][0]
RES1 = [r for r in W1 if r["report_type"] == "resource"]
PRI1 = [r for r in W1 if r["report_type"] == "principal"]
print(f"\n  seq={S1['report_seq']}  host={S1['hostname']}  partial={S1['partial_window']!r}")
print(f"  seen={S1['access_seen']}  kept={S1['access_kept']}  counted={S1['access_counted']}"
      f"  (GET {S1['counted_get']}, POST {S1['counted_post']})  errors_kept={S1['errors_kept']}")
print(f"  distinct: {S1['distinct_resources']} resources, {S1['distinct_principals']} principals"
      f"   __other__ hits: {S1['resources_other']}")

# A REPLAY LOOKS LIKE A SPIKE. The tail DB is on an emptyDir with Read_from_Head true, so a
# `helm upgrade` of the shipper re-reads the whole log file and VictoriaLogs does not
# deduplicate on ingest. Under v3 the replayed reads are absorbed as counts -- but the counters
# attribute the entire replay to whichever window it lands in. A window whose record span is
# far wider than the window itself is a replay, not traffic.
_span = None
if S1.get("min_record_time") and S1.get("max_record_time"):
    _span = (lc._epoch_of(S1["max_record_time"][:19] + "Z")
             - lc._epoch_of(S1["min_record_time"][:19] + "Z"))
_replay = _span is not None and _span > WINDOW_SECONDS
print(f"  record span: {_span}s against a {WINDOW_SECONDS}s window")
if _replay:
    print("   !! WIDER THAN THE WINDOW -- this is a REPLAY (emptyDir tail DB + Read_from_Head),")
    print("      not traffic. Every error and mutation replays as a full record and the whole")
    print("      replay is attributed to this window. Do not read these counts as a spike.")

print("\n  --- resource rows ---")
for r in sorted(RES1, key=lambda r: -int(r.get("requests", 0) or 0)):
    print(f"   {r['resource'][:62]:62} {r['resource_kind']:11} req={r['requests']:>4}"
          f" rd={r['reads']:>4} wr={r['writes']:>4} err={r['errors']:>4} bytes={r['response_bytes']:>8}")
print("  --- principal rows ---")
for r in sorted(PRI1, key=lambda r: -int(r.get("requests", 0) or 0)):
    print(f"   {r['user_principal_name'][:40]:40} req={r['requests']:>4} rd={r['reads']:>4}"
          f" wr={r['writes']:>4} err={r['errors']:>4}")

KINDS_SEEN = sorted({r["resource_kind"] for r in RES1})
print(f"\n  resource_kind seen: {KINDS_SEEN}")
print(f"  all six reachable : {sorted(set(lc.RESOURCE_KINDS) - set(KINDS_SEEN)) or 'yes'}")
print(f"  no /metrics row   : {not any('/metrics' in r['resource'] for r in RES1)}")
REPORT["w1"] = {"window": RUN_WINDOW, "rows": W1, "invariants": BAD, "kinds": KINDS_SEEN}
W1_EPOCH = lc._epoch_of(RUN_WINDOW)


def wait_for_window(start_epoch, lag=None):
    """Wait for the report of the window STARTING at `start_epoch`, and return its rows.

    Cells 13 and 14 must land on SPECIFIC consecutive windows, not merely on "the next
    boundary". At WINDOW_SECONDS=30 a slow cell in between can easily push past a boundary,
    and then zero-carry would be checked against a window whose rows had already decayed --
    a spurious failure that looks exactly like a real one. So target the window explicitly,
    and if we have already overrun it, say so plainly instead of asserting on the wrong data.
    """
    lag = max(8.0, INGEST_LAG_S * 3) if lag is None else lag
    due = start_epoch + WINDOW_SECONDS + lag
    now = time.time()
    if now > due + WINDOW_SECONDS:
        print(f"  !! OVERRAN: window {lc._iso_z(start_epoch)} closed "
              f"{now - due:.0f}s ago and later windows have already flushed. The assertions "
              "below are VOID for this run, not failed -- re-run from cell 11.")
    elif now < due:
        print(f"  waiting {due - now:.0f}s for window {lc._iso_z(start_epoch)}")
        time.sleep(due - now)
    return lc._iso_z(start_epoch), V.report_window(lc._iso_z(start_epoch), since="6h")


## 12 — The oracle diff

Cell 11 asks whether the report is *internally* consistent. This asks whether it is **right**.

The run's own calls are replayed as synthetic records through the deployed Lua, with a tick on
the same boundary, and the array that comes back **is** the expected report. Diffing it against
what VictoriaLogs stored makes full schema coverage a diff rather than a set of hand-written
expectations, and keeps the expectations un-driftable from what ships.

Three field groups are excluded and each for a stated reason: `hostname` and `report_seq`
describe the emitting *process* (the oracle is not the shipper pod); `response_bytes` is not
predictable from the client, which has no `%b`; `min_record_time`/`max_record_time` come from
the real records' own timestamps.

**The window is not expected to hold only this run.** Anything else talking to Polaris counts
too, so the summary counters are compared as a *delta explanation*, while the strict equality
is asserted on the fixture's own resource rows — paths only this run touched.

In [ ]:
DIFF_IGNORE = lc.VOLATILE_FIELDS | {
    "response_bytes",        # no %b client-side; see PLAN section 2.5
    "min_record_time", "max_record_time",
}

_synth = []
for c in ALL_CALLS:
    if not c.get("status"):
        continue          # a call that never completed leaves no access-log line at all
    _synth.append(lc.access_log_record(
        c["method"],
        c.get("actual_path") or lc.full_path(c["api"], c["path"]),
        c["status"],
        user=DRIVE_PRINCIPAL,
    ))

_, _oracle = POLICY.report_windows(_synth, silent_windows=0)
EXPECTED_ROWS = _oracle["w1"]
print(f"  oracle: {len(EXPECTED_ROWS)} rows from {len(_synth)} replayed calls")
print("  oracle invariants:", lc.check_invariants(EXPECTED_ROWS) or "OK")

DIFF = lc.diff_reports(EXPECTED_ROWS, W1, ignore=DIFF_IGNORE)
MIS = lc.report_mismatches(DIFF)
print(f"  diff: {len(DIFF) - len(MIS)} matching fields, {len(MIS)} mismatches\n")

import pandas as pd
if MIS:
    D = pd.DataFrame(MIS)
    D["key"] = D["key"].str.slice(-52)
    print(D.to_string(index=False))

# STRICT on the fixture's own resources. Everything else in the window may be someone else's
# traffic; these paths are ours alone, so a mismatch here is the pipeline, not the neighbours.
OURS = [m for m in MIS if FX.cat in str(m["key"])]
print(f"\n  mismatches on THIS RUN'S fixture resources: {len(OURS)}")
for m in OURS:
    print(f"   {m['report_type']:9} {str(m['key'])[-52:]:52} {m['field']:16}"
          f" expected={m['expected']!r} actual={m['actual']!r}")

# The one row the ordering in cell 5 exists to produce.
_tbl_key = f"/api/catalog/v1/{FX.cat}/namespaces/{FX.ns}/tables/{FX.tbl}"
_row = next((r for r in RES1 if r["resource"] == _tbl_key), None)
if _row is None:
    print(f"\n  !! no resource row for {_tbl_key} -- the fixture table was never read")
else:
    _nz = {k: _row[k] for k in ("requests", "reads", "writes", "errors")}
    print(f"\n  fixture table row: {_nz}")
    print("  all four non-zero:", all(int(v) > 0 for v in _nz.values()),
          "  (errors=0 is EXPECTED if the 403 client could not be provisioned)")
REPORT["diff"] = {"mismatches": MIS, "ours": OURS}


## 13 — Window 2: the counters reset, and a quiet resource still reports

Two properties that only exist *across* windows, and both are about a scheduled job rather than
about Polaris:

- **counters reset** — window N+1 does not carry N's totals forward;
- **zero-carry** — a resource that was active in N emits an explicit `0` in N+1.

Zero-carry is what lets a dashboard tell *no traffic* from *no shipper*. Without it a series
simply disappears, and disappearing is what an outage looks like too.

This cell **stays deliberately silent** for one whole window. Nothing is driven.

In [ ]:
W2_START, W2 = wait_for_window(W1_EPOCH + WINDOW_SECONDS)
print(f"  window {W2_START}: {len(W2)} rows")
if not W2:
    raise RuntimeError(
        f"no report at all for {W2_START}. Either the shipper stopped, or a tick arrived late "
        "enough that the filter's window index jumped and this window was never reported -- "
        "cell 14 distinguishes them by looking for a gap between consecutive window_starts.")

BAD2 = lc.check_invariants(W2)
print("  invariants:", "OK" if not BAD2 else BAD2)

S2 = [r for r in W2 if r["report_type"] == "summary"][0]
RES2 = {r["resource"]: r for r in W2 if r["report_type"] == "resource"}

print(f"  seq {S1['report_seq']} -> {S2['report_seq']}"
      f"   {'OK' if int(S2['report_seq']) == int(S1['report_seq']) + 1 else '<-- NOT +1'}")
print(f"  same host: {S2['hostname'] == S1['hostname']}"
      f"   {'' if S2['hostname'] == S1['hostname'] else '<-- THE SHIPPER RESTARTED: counts are void'}")
print(f"  counters reset: access_seen={S2['access_seen']} (window 1 saw {S1['access_seen']})"
      f"   {'OK' if int(S2['access_seen']) < int(S1['access_seen']) else '<-- did not reset'}")
print(f"  partial_window: {S2['partial_window']!r}   (expect 'false' -- window 1 may be 'true')")

_active = {r["resource"] for r in RES1 if int(r.get("requests", 0) or 0) > 0}
_carried = _active & set(RES2)
_zeroed = {k for k in _carried if int(RES2[k].get("requests", 0) or 0) == 0}
print(f"\n  ZERO-CARRY: {len(_active)} resources active in window 1")
print(f"              {len(_carried)} carried into window 2"
      f"   {'OK' if _carried == _active else '<-- MISSING: ' + str(sorted(_active - _carried))[:200]}")
print(f"              {len(_zeroed)} of them report an explicit 0"
      f"   {'OK' if _zeroed == _carried else '<-- some carried a non-zero count'}")
REPORT["w2"] = {"window": W2_START, "rows": W2, "carried": sorted(_carried)}


## 14 — Window 3: carry decay, and the tick is not the period

The other half of zero-carry: a row that stayed at `0` is **not** carried a third time.
Without decay, every key ever seen is reported forever and the report grows without bound.

And the property the whole design turns on, asserted end to end at last: the dummy INPUT ticks
every 30 s, so a `WINDOW_SECONDS` of 1800 sees **60 ticks per report**. If the filter emitted on
the tick rather than on the boundary there would be 60 reports per window. Count them.

Silent again — nothing is driven.

In [ ]:
W3_START, W3 = wait_for_window(W1_EPOCH + 2 * WINDOW_SECONDS)
RES3 = {r["resource"] for r in W3 if r["report_type"] == "resource"}
S3 = [r for r in W3 if r["report_type"] == "summary"][0]
print(f"  window {W3_START}: {len(W3)} rows")
print("  invariants:", lc.check_invariants(W3) or "OK")

_stale = {k for k in REPORT["w2"]["carried"]}
_still = _stale & RES3
print(f"\n  CARRY DECAY: {len(_stale)} rows sat at 0 through window 2")
print(f"               {len(_still)} of them are still being reported"
      f"   {'OK -- decayed' if not _still else '<-- CARRIED A THIRD TIME: ' + str(sorted(_still))[:200]}")

# ONE REPORT PER WINDOW PER HOST, whatever the tick rate.
_all = V.reports(report_type="summary", since="6h", limit=5000)
_by = {}
for r in _all:
    _by.setdefault((r.get("hostname"), r.get("window_start")), 0)
    _by[(r.get("hostname"), r.get("window_start"))] += 1
_dupes = {k: n for k, n in _by.items() if n > 1}
_ticks_per_window = WINDOW_SECONDS / (TICK_SECONDS or 30)
print(f"\n  TICK != PERIOD: the dummy INPUT ticks every {TICK_SECONDS}s, so ~"
      f"{_ticks_per_window:.0f} ticks fall inside each {WINDOW_SECONDS}s window")
print(f"  summaries per (host, window): {sorted(set(_by.values()))}"
      f"   {'OK -- exactly one' if set(_by.values()) == {1} else '<-- ' + str(_dupes)[:200]}")

# A SKIPPED WINDOW is the failure mode at small WINDOW_SECONDS: if a tick lands late the
# filter's window index jumps by two, it reports the window it was holding, and the one in
# between never existed. Visible as a gap between consecutive window_starts.
_starts = sorted({lc._epoch_of(r["window_start"]) for r in _all if r.get("window_start")})
_gaps = [(a, b) for a, b in zip(_starts, _starts[1:]) if b - a != WINDOW_SECONDS]
print(f"  consecutive windows: {len(_starts)}   gaps: "
      f"{'none' if not _gaps else [(lc._iso_z(a), lc._iso_z(b)) for a, b in _gaps]}")
if _gaps:
    print(f"   -> a gap means a tick arrived late enough that the window index jumped: the "
          f"filter reported the window it was holding and the one in between never existed. "
          f"At {WINDOW_SECONDS}s windows with a {TICK_SECONDS}s tick "
          f"({_ticks_per_window:.0f} ticks per window) that is a FINDING, not the setup.")
print(f"\n  raw tick leak (must be 0): {V.count(vlogs.tick_leak(), start='6h')}")
REPORT["w3"] = {"window": W3_START, "rows": W3, "decayed": not _still, "gaps": _gaps}


## 15 — Findings


In [11]:
def q(x):
    return "unknown" if x is None else x


report = f"""# Log coverage — run {RUN}

Generated by `polaris_log_coverage.ipynb` on {STARTED.date().isoformat()}.
Correlation mode **{CORRELATION}**; drove as **{DRIVE_PRINCIPAL}**.
Expected column computed from `{POLICY.source if POLICY else 'UNAVAILABLE'}`
(sha256 `{POLICY.sha256[:16] if POLICY else '-'}`).

## Validity

| | |
|---|---|
| shipper pod before | `{SHIPPER_BEFORE}` |
| shipper pod after | `{q(SHIPPER_AFTER)}` |
| Polaris replicas before / after | `{HPA_BEFORE}` / `{q(HPA_AFTER)}` |
| duplicate records (shipper replay) | {DUPES} |
| ingest lag, first record | {INGEST_LAG_S:.1f}s |
| numeric `http_status` filters | {q(NUMERIC_DETAIL)} |
| retention policy deployed? | {q(POLICY_LIVE)} — {POLICY_LIVE_DETAIL.splitlines()[0]} |
| records per call | {len(STORED) / max(1, len(ALL_CALLS)):.1f} |

{"**THIS RUN IS INVALID** — Polaris scaled mid-run." if HPA_MOVED else ""}
{"**Dedup expectations are VOID** — the shipper restarted mid-run." if SHIPPER_MOVED else ""}

## The seven questions

1. **Distinct WARN messages** — {len(DISTINCT_WARNS)} seen:
{chr(10).join(f"   - `{lvl}` x{n}: {msg}" for (lvl, msg), n in DISTINCT_WARNS.items()) or "   - none, so the Deprecated Config exclusion still has nothing to match narrowly."}
2. **Where a PUT's body is logged** — {len(priv_hits)} record(s) in VictoriaLogs carry the
   granted privilege name. eventListener `events` table: {"%s rows in schema %s" % (EVENTS["rows"], EVENTS["schema"]) if EVENTS["reachable"] else EVENTS["note"]}.
3. **Do stack traces survive?** — {len(EXC)} of {len(WARNS)} WARN/ERROR records carried an
   exception object.
4. **Does `mdc.requestId` join reliably?** — {CORRELATION}; {len([k for k in BY_RID if k])} distinct
   ids recovered from {len(ALL_CALLS)} calls.
5. **The v2 audit hole, and whether v3 closed it** — {int((M.expected == "drop").sum())} of {len(M)}
   calls leave no individual record; every one of them is COUNTED into the window aggregate.
   Management POSTs kept: {int((M[M.path.str.contains("/api/management", na=False) & (M.method == "POST")].expected == "keep").sum())}
   of {len(M[M.path.str.contains("/api/management", na=False) & (M.method == "POST")])} — under v2 this was zero, which is what made a
   credential reset untraceable.
6. **Volume** — {len(STORED)} records stored for {len(ALL_CALLS)} calls;
   `fluentbit_filter_drop_records_total` delta {q(DROP_DELTA)}.
7. **Client-side latency** (there is no `%D`, so this is the only source):
   median {M.ms.median():.0f} ms, p95 {M.ms.quantile(0.95):.0f} ms, max {M.ms.max():.0f} ms.

## Not callable on this cluster, with the reason

{chr(10).join(f"- **{k}** — {v}" for k, v in NOT_CALLABLE.items())}

## Coverage matrix

```
{M.to_string(index=False)}
```

## The scheduled flush report

A `dummy` INPUT tagged `polaris.report` ticks every 30s into the same Lua filter; on each
`{WINDOW_SECONDS}`s boundary the filter replaces the tick with an ARRAY of records and resets
its counters. **The tick rate is not the report period.** Schema v1, three record types on one
envelope, on their own stream `{{app="polaris-shipper-report", level="REPORT"}}`.

| | |
|---|---|
| `window_seconds`, as deployed | {WINDOW_SECONDS} |
| window that saw this run | `{REPORT["w1"]["window"]}` |
| rows in it | {len(REPORT["w1"]["rows"])} |
| schema invariants | {"OK" if not REPORT["w1"]["invariants"] else REPORT["w1"]["invariants"]} |
| `resource_kind` values seen | {REPORT["w1"]["kinds"]} |
| oracle diff, all fields | {len(REPORT["diff"]["mismatches"])} mismatches |
| oracle diff, THIS run's fixture resources | {len(REPORT["diff"]["ours"])} mismatches |
| zero-carry (window 2) | {len(REPORT["w2"]["carried"])} rows carried at 0 |
| carry decay (window 3) | {"decayed" if REPORT["w3"]["decayed"] else "STILL CARRIED"} |
| window gaps (a skipped window) | {REPORT["w3"]["gaps"] or "none"} |

The margin equality — `sum(resource.requests) == sum(principal.requests) ==
access_seen - parse_errors` — is the schema's only real self-check, because `errors`
deliberately overlaps `reads`/`writes`. Note that `access_kept + access_counted ==
access_seen` is **tautological in the filter** (`build_report` computes `access_kept` by
subtraction) and is worth asserting only end to end, where it is a transport check.

**A startup blind spot, measured:** `report_tick` opens its first window on the FIRST tick, and
`count_record()` returns immediately while `counts` is nil. Records processed between shipper
start and that first tick are routed by the policy correctly but appear in NO report. Bounded
by the tick interval (30s). The window they land in is flagged `partial_window: true`.
"""
out = pathlib.Path("doc-log-coverage-results.md")
out.write_text(report, encoding="utf-8")
print(f"  wrote {out} ({len(report):,} bytes)")
print("\n  Next: update MEMORY.md 'Now', put the numbers in .memory/roadmap.md, anything")
print("  not to be trusted in .memory/active-issues.md, and the wrong turns in")
print("  .memory/sessions/<date>-log-coverage.md. Then commit -- one task, one commit.")

  wrote doc-log-coverage-results.md (23,708 bytes)

  Next: update MEMORY.md 'Now', put the numbers in .memory/roadmap.md, anything
  not to be trusted in .memory/active-issues.md, and the wrong turns in
  .memory/sessions/<date>-log-coverage.md. Then commit -- one task, one commit.
